In [1]:
import pymupdf

doc = pymupdf.open("../data/text_files/Reliance.pdf")

print(doc.page_count)

174


In [2]:
text = doc[30].get_text()
print(text[:200])

Dear Shareholders,
Nearly fifty years ago, our visionary 
founder, Shri Dhirubhai Ambani, 
embarked on a bold mission — 
to prove that India could build a 
world-class enterprise founded on 
innovatio


In [3]:
text = doc[30].get_text()
lines = text.split("\n")
print(lines[:10])

['Dear Shareholders,', 'Nearly fifty years ago, our visionary ', 'founder, Shri Dhirubhai Ambani, ', 'embarked on a bold mission — ', 'to prove that India could build a ', 'world-class enterprise founded on ', 'innovation, integrity, and ambition. ', 'That belief became Reliance. And ', 'over the decades, Reliance has ', 'grown from an idea into one of the ']


In [4]:
text = doc[30].get_text()
lines = text.split("\n")

for line in lines[:10]:
    print(line)

Dear Shareholders,
Nearly fifty years ago, our visionary 
founder, Shri Dhirubhai Ambani, 
embarked on a bold mission — 
to prove that India could build a 
world-class enterprise founded on 
innovation, integrity, and ambition. 
That belief became Reliance. And 
over the decades, Reliance has 
grown from an idea into one of the 


In [5]:
for page_num in range(27, 30):
    text = doc[page_num].get_text()
    lines = text.split("\n")
    for line in lines[:5]:
        print(page_num, "|", line)

27 | Integrated Annual Report
27 | 2024-25
27 | Realising
27 | Aspirations
27 | Accessibility
28 | TABLE OF CONTENTS
28 | REPORTING SUITE  
28 | 2024-25
28 | RIL’s Annual Reporting suite brings 
28 | together the financial, non-financial, 
29 | India’s top retailer, delivering exceptional reach, providing 
29 | quality products and superior customer experience 
29 | through a unified network of stores and cutting-edge 
29 | digital platforms.
29 | Consumption Baskets


In [6]:
counts = {}

for page_num in range(27, 30):
    text = doc[page_num].get_text()
    lines = text.split("\n")
    for line in lines:
        if line in counts:
            counts[line] = counts[line] + 1
        else:
            counts[line] = 1

print(counts)

{'Integrated Annual Report': 2, '2024-25': 3, 'Realising': 2, 'Aspirations': 2, 'Accessibility': 2, 'Reliability': 1, 'Variety': 2, 'Mobility': 2, 'Connectivity': 2, 'Responsibility': 2, 'Sustainability': 2, '': 3, 'TABLE OF CONTENTS': 1, 'REPORTING SUITE  ': 1, 'RIL’s Annual Reporting suite brings ': 1, 'together the financial, non-financial, ': 1, 'risk, and sustainability performance for ': 1, 'the year.': 1, 'Corporate Overview': 1, '2 ': 1, 'Reliance at a Glance': 1, '3 ': 1, 'Stakeholder Value Creation': 1, '4 ': 1, 'Chairman and Managing Director’s Statement ': 1, '6 ': 1, '10-year Financial Highlights': 1, 'Management Discussion and Analysis': 1, '7 ': 1, 'Financial Performance and Review': 1, '\t': 1, 'Business Overview': 1, '9 ': 1, '\t Retail': 1, '12 ': 1, ' Digital Services': 1, '15 ': 1, ' Media and Entertainment': 1, '19 ': 1, ' Oil to Chemicals': 1, '22 ': 1, ' Oil and Gas': 1, '25 ': 1, 'Risk and Governance': 1, '28 ': 1, 'Major Awards and Recognitions': 1, 'Integrated

In [13]:
import pymupdf
import json

# ---------- PARSING ----------

BOILERPLATE = {
    "Reliance Industries Limited",
    "Integrated Annual Report 2024-25",
}


def clean_page(text):
    lines = [l for l in text.split("\n") if l.strip() not in BOILERPLATE]
    text = "\n".join(lines)
    text = text.replace("(C in crore)", "(₹ in crore)")
    text = text.replace("\u0007", "")
    while "  " in text:
        text = text.replace("  ", " ")
    while "\n\n\n" in text:
        text = text.replace("\n\n\n", "\n\n")
    return text.strip()


def parse_pdf(path, company, year, start_page=27):
    doc = pymupdf.open(path)
    pages = []
    for page_num in range(start_page, doc.page_count):
        raw = doc[page_num].get_text()
        text = clean_page(raw)
        if len(text.split()) < 30:      # drop near-empty pages
            continue
        pages.append({
            "text": text,
            "company": company,
            "fiscal_year": year,
            "page": page_num,
        })
    return pages


# ---------- CHUNKING ----------

def chunk_text(text, size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


def chunk_pages(pages, size=1000, overlap=200):
    all_chunks = []
    for page in pages:
        pieces = chunk_text(page["text"], size, overlap)
        for i, piece in enumerate(pieces):
            all_chunks.append({
                "text": piece,
                "company": page["company"],
                "fiscal_year": page["fiscal_year"],
                "page": page["page"],
                "chunk_index": i,
            })
    return all_chunks


# ---------- RUN ----------

pages = parse_pdf("../data/text_files/Reliance.pdf", "Reliance", "2024-25")
print(f"{len(pages)} pages parsed")

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks created")

print(chunks[10]["text"])

with open("reliance_chunks.json", "w") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

146 pages parsed
1276 chunks created
Dear Shareholders,
Nearly fifty years ago, our visionary 
founder, Shri Dhirubhai Ambani, 
embarked on a bold mission — 
to prove that India could build a 
world-class enterprise founded on 
innovation, integrity, and ambition. 
That belief became Reliance. And 
over the decades, Reliance has 
grown from an idea into one of the 
world’s most admired enterprises — 
a symbol of India’s entrepreneurial 
spirit and its limitless potential.
Shri. Mukesh D. Ambani
Chairman and Managing Director, 
Today, Reliance is not just a company. 
It is a national institution that powers 
opportunity, progress, and prosperity 
for 1.45 billion Indians. Our journey has 
been extraordinary — transforming 
traditional industries, building global-
scale capabilities, creating entirely new 
markets, and touching the everyday 
lives of our fellow citizens. But we have 
never paused to celebrate the past. 
Instead, each milestone has inspired 
us to aim higher, work harder,

In [10]:
page_text = doc[30].get_text()
lines = page_text.split("\n")

print(lines[0])
print(lines[1])
print(lines[2])

Dear Shareholders,
Nearly fifty years ago, our visionary 
founder, Shri Dhirubhai Ambani, 


In [11]:
for line in lines:
    print(line)

Dear Shareholders,
Nearly fifty years ago, our visionary 
founder, Shri Dhirubhai Ambani, 
embarked on a bold mission — 
to prove that India could build a 
world-class enterprise founded on 
innovation, integrity, and ambition. 
That belief became Reliance. And 
over the decades, Reliance has 
grown from an idea into one of the 
world’s most admired enterprises — 
a symbol of India’s entrepreneurial 
spirit and its limitless potential.
Shri. Mukesh D. Ambani
Chairman and Managing Director, 
Reliance Industries Limited
Today, Reliance is not just a company. 
It is a national institution that powers 
opportunity, progress, and prosperity 
for 1.45 billion Indians. Our journey has 
been extraordinary — transforming 
traditional industries, building global-
scale capabilities, creating entirely new 
markets, and touching the everyday 
lives of our fellow citizens. But we have 
never paused to celebrate the past. 
Instead, each milestone has inspired 
us to aim higher, work harder, and lead